# Fraud Score Retro Evaluation

Goal: Evaluate how well a fraud score would have prevented fraud and bad outcomes using historical data.

## Import libraries & Data 

In [1]:
import sys
print(sys.executable)

/Users/spencerwillard/code/local/fraud_score_retro_eval/.venv/bin/python


In [2]:
# Import necesary libraries
import pandas as pd
import os

In [3]:
# Confirm Current Directory 
print("Working directory:", os.getcwd()) 
print("Files here:", os.listdir())

Working directory: /Users/spencerwillard/code/local/fraud_score_retro_eval/notebooks
Files here: ['fraud_analysis.ipynb']


In [4]:
# Import Data Sets

scores = pd.read_csv("../data/scores.csv", sep=",")
outcomes = pd.read_csv("../data/outcomes.csv", sep=",")

## Inspect the Data

In [5]:
# Inspect Scores File

print("Scores File")
scores.head()

Scores File


,"application_id,fraud_score"
0,"1,001,320"
1,"1,002,910"
2,"1,003,410"
3,"1,004,780"
4,"1,005,250"


In [6]:
# Inspect Outcomes File

print("Outcomes File")
outcomes.head()

Outcomes File


,"application_id,outcome"
0,"1001,good_standing"
1,"1002,fraud"
2,"1003,good_standing"
3,"1004,charged_off"
4,"1005,good_standing"


In [7]:
# Inspect Scores File 
scores.head(10) 

,"application_id,fraud_score"
0,"1,001,320"
1,"1,002,910"
2,"1,003,410"
3,"1,004,780"
4,"1,005,250"
5,"1,006,340"
6,"1,007,870"
7,"1,008,290"
8,"1,009,730"
9,"1,010,310"


In [8]:
scores.shape

(50, 1)

In [9]:
scores.columns

Index(['application_id,fraud_score'], dtype='str')

In [10]:
scores.dtypes

application_id,fraud_score    str
dtype: object

In [11]:
scores.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 1 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   application_id,fraud_score  50 non-null     str  
dtypes: str(1)
memory usage: 532.0 bytes


In [12]:
outcomes.head(10)

,"application_id,outcome"
0,"1001,good_standing"
1,"1002,fraud"
2,"1003,good_standing"
3,"1004,charged_off"
4,"1005,good_standing"
5,"1006,good_standing"
6,"1007,fraud"
7,"1008,good_standing"
8,"1009,charged_off"
9,"1010,good_standing"


In [13]:
outcomes.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 1 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   application_id,outcome  50 non-null     str  
dtypes: str(1)
memory usage: 532.0 bytes


## Clean up before validation

In [14]:
scores["raw"] = scores["application_id,fraud_score"].astype(str).str.replace(",", "", regex=False)

In [15]:
scores["application_id"] = scores["raw"].str[:4]
scores["fraud_score"] = scores["raw"].str[4:]

In [16]:
scores = scores.drop(columns=["application_id,fraud_score", "raw"])

In [17]:
scores["application_id"] = scores["application_id"].astype(int)
scores["fraud_score"] = scores["fraud_score"].astype(int)

In [18]:
scores.head()

,application_id,fraud_score
0,1001,320
1,1002,910
2,1003,410
3,1004,780
4,1005,250


In [19]:
outcomes.head()

,"application_id,outcome"
0,"1001,good_standing"
1,"1002,fraud"
2,"1003,good_standing"
3,"1004,charged_off"
4,"1005,good_standing"


In [20]:
outcomes.columns

Index(['application_id,outcome'], dtype='str')

In [21]:
outcomes[["application_id", "outcome"]] = outcomes["application_id,outcome"].str.split(",", n=1, expand=True)

In [22]:
outcomes = outcomes.drop(columns=["application_id,outcome"])

In [23]:
outcomes.head()

,application_id,outcome
0,1001,good_standing
1,1002,fraud
2,1003,good_standing
3,1004,charged_off
4,1005,good_standing


In [24]:
outcomes["application_id"] = outcomes["application_id"].astype(int)

In [25]:
outcomes.dtypes

application_id    int64
outcome             str
dtype: object

## Validate the Data

In [26]:
# Any missing data in scores?
scores.isna().sum()

application_id    0
fraud_score       0
dtype: int64

In [27]:
# Any missing data in outcomes? 
outcomes.isna().sum()

application_id    0
outcome           0
dtype: int64

In [28]:
#Make sure application_id is unique for both DataFrames
print(scores["application_id"].nunique()) 
print(outcomes["application_id"].nunique())


50
50


In [29]:
# Does the unique value match the df length for both? 
print(len(scores))
print(len(outcomes))

50
50


In [30]:
# Understand Fraud Score Distribution
scores["fraud_score"].describe()

count     50.000000
mean     507.800000
std      261.475718
min      250.000000
25%      300.000000
50%      335.000000
75%      767.500000
max      920.000000
Name: fraud_score, dtype: float64

In [31]:
# Understand outcome distribution 
outcomes["outcome"].value_counts()


outcome
good_standing    31
fraud            10
charged_off       9
Name: count, dtype: int64

In [32]:
outcomes["outcome"].value_counts(normalize=True) * 100 

outcome
good_standing    62.0
fraud            20.0
charged_off      18.0
Name: proportion, dtype: float64

## Merge files 

In [33]:
df = scores.merge(outcomes, how="inner", on="application_id" )

In [34]:
df.shape

(50, 3)

In [35]:
df.head(10)

,application_id,fraud_score,outcome
0,1001,320,good_standing
1,1002,910,fraud
2,1003,410,good_standing
3,1004,780,charged_off
4,1005,250,good_standing
5,1006,340,good_standing
6,1007,870,fraud
7,1008,290,good_standing
8,1009,730,charged_off
9,1010,310,good_standing


In [36]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   application_id  50 non-null     int64
 1   fraud_score     50 non-null     int64
 2   outcome         50 non-null     str  
dtypes: int64(2), str(1)
memory usage: 1.3 KB


In [37]:
# Double check we still have unique application_id
print("Total Unique application_id:", df["application_id"].nunique())

Total Unique application_id: 50


## Data Summary & Examples

In [38]:
# "Does the score separate fraud?"
df.groupby("outcome")["fraud_score"].describe().sort_values(by="count", ascending=False).round(2)

,count,mean,std,min,25%,50%,75%,max
outcome,,,,,,,,
good_standing,31.0,309.52,31.66,250.0,292.5,310.0,325.00,410.0
fraud,10.0,902.00,17.03,870.0,892.5,907.5,913.75,920.0
charged_off,9.0,752.78,23.86,720.0,735.0,750.0,770.00,790.0


In [39]:
# High scoring fraud examples
df.sort_values("fraud_score", ascending=False).head(10)

,application_id,fraud_score,outcome
11,1012,920,fraud
47,1048,920,fraud
27,1028,915,fraud
42,1043,910,fraud
1,1002,910,fraud
21,1022,905,fraud
37,1038,900,fraud
32,1033,890,fraud
16,1017,880,fraud
6,1007,870,fraud


In [40]:
# Low scoring good apps  
df.sort_values("fraud_score").head(10)

,application_id,fraud_score,outcome
4,1005,250,good_standing
41,1042,260,good_standing
20,1021,270,good_standing
36,1037,275,good_standing
10,1011,280,good_standing
25,1026,285,good_standing
7,1008,290,good_standing
31,1032,290,good_standing
46,1047,295,good_standing
17,1018,295,good_standing


In [41]:
df[df["outcome"] == "charged_off"] \
    .sort_values("fraud_score", ascending=False) \
    .head(10)

,application_id,fraud_score,outcome
19,1020,790,charged_off
3,1004,780,charged_off
24,1025,770,charged_off
14,1015,760,charged_off
30,1031,750,charged_off
35,1036,740,charged_off
45,1046,735,charged_off
8,1009,730,charged_off
40,1041,720,charged_off


## Fraud Prevention Analysis

### Fraud Performance at >= 700

In [42]:
# Set score cutoff threshold
cutoff_700 = 700 

In [43]:
# "What ACTION should we take based on the score?"
df["predicted_fraud"] = df["fraud_score"] >= cutoff_700

In [44]:
df["actual_fraud"] = df["outcome"] == "fraud"

In [45]:
ct_700 = pd.crosstab(df["predicted_fraud"], df["actual_fraud"])

ct_700.index = ["Predicted: Clear", "Predicted: Fraud"]
ct_700.columns = ["Actual: Clear", "Actual: Fraud"]

ct_700["Total"] = ct_700.sum(axis=1)
ct_700.loc["Total"] = ct_700.sum()

ct_700

,Actual: Clear,Actual: Fraud,Total
Predicted: Clear,31,0,31
Predicted: Fraud,9,10,19
Total,40,10,50


In [46]:
# Set confusion metrics 
true_negative = ct_700.loc["Predicted: Clear", "Actual: Clear"]
false_negative = ct_700.loc["Predicted: Clear", "Actual: Fraud"]
false_positive = ct_700.loc["Predicted: Fraud", "Actual: Clear"]
true_positive = ct_700.loc["Predicted: Fraud", "Actual: Fraud"]

In [47]:
# Determine Precision of Apps Scored >= 700 
precision = true_positive / (true_positive + false_positive)
print(f"Precision >= {cutoff_700}:", precision.round(2))

Precision >= 700: 0.53


In [48]:
# Determine Recall of Apps Scored >= 700 
recall = (true_positive) / (false_negative + true_positive)
print(f"Recall >= 700: {cutoff_700}", recall.round(2))

Recall >= 700: 700 1.0


### Fraud performance at >= 800 

In [49]:
# Set cutoff to >= 800 
cutoff_800 = 800

In [50]:
# "What ACTION should we take based on the score?"
df["predicted_fraud"] = df["fraud_score"] >= cutoff_800

In [51]:
df["actual_fraud"] = df["outcome"] == "fraud"

In [52]:
ct_800 = pd.crosstab(df["predicted_fraud"], df["actual_fraud"])

ct_800.index = ["Predicted: Clear", "Predicted: Fraud"]
ct_800.columns = ["Actual: Clear", "Actual: Fraud"]

ct_800["Total"] = ct_800.sum(axis=1)
ct_800.loc["Total"] = ct_800.sum()

ct_800

,Actual: Clear,Actual: Fraud,Total
Predicted: Clear,40,0,40
Predicted: Fraud,0,10,10
Total,40,10,50


In [69]:
print(ct_800.index)
print(ct_800.columns)

Index(['Predicted: Clear', 'Predicted: Fraud', 'Total'], dtype='str')
Index(['Actual: Clear', 'Actual: Fraud', 'Total'], dtype='str')


In [53]:
# Set confusion metrics 
true_negative = ct_800.loc["Predicted: Clear", "Actual: Clear"]
false_negative = ct_800.loc["Predicted: Clear", "Actual: Fraud"]
false_positive = ct_800.loc["Predicted: Fraud", "Actual: Clear"]
true_positive = ct_800.loc["Predicted: Fraud", "Actual: Fraud"]

In [54]:
# Determine Precision of Apps Scored >= 800 
precision = true_positive / (true_positive + false_positive)
print(f"Precision >= {cutoff_800}:", precision.round(2))

Precision >= 800: 1.0


In [55]:
# Determine Recall of Apps Scored >= 800 
recall = (true_positive) / (false_negative + true_positive)
print(f"Recall >= {cutoff_800}: {recall}")

Recall >= 800: 1.0


## Loss Prevention Analysis

### Loss Performance at >= 800

In [87]:
df.groupby("outcome")["fraud_score"].describe().sort_values(by="count", ascending=False).round(2)

,count,mean,std,min,25%,50%,75%,max
outcome,,,,,,,,
good_standing,31.0,309.52,31.66,250.0,292.5,310.0,325.00,410.0
fraud,10.0,902.00,17.03,870.0,892.5,907.5,913.75,920.0
charged_off,9.0,752.78,23.86,720.0,735.0,750.0,770.00,790.0


In [56]:
df["predicted_bad"] = df["fraud_score"] >= cutoff_800

In [57]:
df["actual_bad"] = df["outcome"].isin(["fraud", "charged_off"])

In [58]:
df.head(10)

,application_id,fraud_score,outcome,predicted_fraud,actual_fraud,predicted_bad,actual_bad
0,1001,320,good_standing,False,False,False,False
1,1002,910,fraud,True,True,True,True
2,1003,410,good_standing,False,False,False,False
3,1004,780,charged_off,False,False,False,True
4,1005,250,good_standing,False,False,False,False
5,1006,340,good_standing,False,False,False,False
6,1007,870,fraud,True,True,True,True
7,1008,290,good_standing,False,False,False,False
8,1009,730,charged_off,False,False,False,True
9,1010,310,good_standing,False,False,False,False


In [93]:
# Create confusion matrix for bad labels
ct_800_loss = pd.crosstab(df["predicted_bad"], df["actual_bad"])

#Clean up index & column names
ct_800_loss.index = ["Predicted: Clear", "Predicted: Bad"]
ct_800_loss.columns = ["Actual: Clear", "Actual: Bad"]

# Add totals 
ct_800_loss["Total"] = ct_800_loss.sum(axis=1)
ct_800_loss.loc["Total"] = ct_800_loss.sum()

print(ct_800_loss)

                  Actual: Clear  Actual: Bad  Total
Predicted: Clear             31            9     40
Predicted: Bad                0           10     10
Total                        31           19     50


In [92]:
print(ct_800_loss.index.tolist()) 
print(ct_800_loss.columns.tolist()) 

['Predicted: Clear', 'Predicted: Bad', 'Total']
['Actual: Clear', 'Actual: Bad', 'Total']


In [91]:
true_negative = ct_800_loss.loc["Predicted: Clear", "Actual: Clear"]
false_negative = ct_800_loss.loc["Predicted: Clear", "Actual: Bad"]
false_positive = ct_800_loss.loc["Predicted: Bad", "Actual: Clear"]
true_positive = ct_800_loss.loc["Predicted: Bad", "Actual: Bad"]

In [94]:
precision = true_positive / (true_positive + false_positive)
print(f"Precision >= {cutoff_800}: {precision.round(2)}")

Precision >= 800: 1.0


In [97]:
recall = true_positive / (true_positive + false_negative)
print(f"Recall >= {cutoff_800}: {recall.round(2)}")

Recall >= 800: 0.53


### Checking loss prevention business case at >= 800 

In [103]:
# "What ACTION should we take based on the score?"
df["predicted_bad"] = df["fraud_score"] >= cutoff_700

In [106]:
# Create confusion matrix for bad labels
ct_700_loss = pd.crosstab(df["predicted_bad"], df["actual_bad"])

#Clean up index & column names
ct_700_loss.index = ["Predicted: Clear", "Predicted: Bad"]
ct_700_loss.columns = ["Actual: Clear", "Actual: Bad"]

# Add totals 
ct_700_loss["Total"] = ct_700_loss.sum(axis=1)
ct_700_loss.loc["Total"] = ct_700_loss.sum()

print(ct_700_loss)

                  Actual: Clear  Actual: Bad  Total
Predicted: Clear             31            0     31
Predicted: Bad                0           19     19
Total                        31           19     50


In [112]:
true_negative = ct_700_loss.loc["Predicted: Clear", "Actual: Clear"]
false_negative = ct_700_loss.loc["Predicted: Clear", "Actual: Bad"]
false_positive = ct_700_loss.loc["Predicted: Bad", "Actual: Clear"]
true_positive = ct_700_loss.loc["Predicted: Bad", "Actual: Bad"]

In [123]:
print("For Losss Prevention Use Case") 
precision = true_positive / (true_positive + false_positive)
print(f"Precision >= {cutoff_700}: {precision.round(2)}")

For Losss Prevention Use Case
Precision >= 700: 1.0


In [124]:
print("For Losss Prevention Use Case") 

recall = true_positive / (true_positive + false_negative)
print(f"Recall >= {cutoff_800}: {recall.round(2)}")

For Losss Prevention Use Case
Recall >= 800: 1.0


In [116]:
df.sort_values("fraud_score")

,application_id,fraud_score,outcome,predicted_fraud,actual_fraud,predicted_bad,actual_bad
4,1005,250,good_standing,False,False,False,False
41,1042,260,good_standing,False,False,False,False
20,1021,270,good_standing,False,False,False,False
36,1037,275,good_standing,False,False,False,False
10,1011,280,good_standing,False,False,False,False
25,1026,285,good_standing,False,False,False,False
7,1008,290,good_standing,False,False,False,False
31,1032,290,good_standing,False,False,False,False
46,1047,295,good_standing,False,False,False,False
17,1018,295,good_standing,False,False,False,False


In [118]:
print(df.groupby("outcome")["fraud_score"].min()) 
print(df.groupby("outcome")["fraud_score"].max())

outcome
charged_off      720
fraud            870
good_standing    250
Name: fraud_score, dtype: int64
outcome
charged_off      790
fraud            920
good_standing    410
Name: fraud_score, dtype: int64


## Create Performance Table for Comparison

In [126]:
# Create variables for table

fraud_700_precision = 0.53
fraud_700_recall = 1.0

fraud_800_precision = 1.00
fraud_800_recall = 1.0

loss_700_precision = 1.0
loss_700_recall = 1.0

loss_800_precision = 1.0
loss_800_recall = 0.53

In [ ]:
# Create the comparison DataFrame
comparison = pd.DataFrame([
    {
        "Use Case": "Fraud Prevention",
        "Cutoff": 700,
        "Precision": fraud_700_precision,
        "Recall": fraud_700_recall
    },
    {
        "Use Case": "Fraud Prevention",
        "Cutoff": 800,
        "Precision": fraud_800_precision,
        "Recall": fraud_800_recall
    },
    {
        "Use Case": "Loss Prevention",
        "Cutoff": 700,
        "Precision": loss_700_precision,
        "Recall": loss_700_recall
    },
    {
        "Use Case": "Loss Prevention",
        "Cutoff": 800,
        "Precision": loss_800_precision,
        "Recall": loss_800_recall
    }
])

comparison = comparison.round(2)
comparison

,Use Case,Cutoff,Precision,Recall
0,Fraud Prevention,700,0.53,1.00
1,Fraud Prevention,800,1.00,1.00
2,Loss Prevention,700,1.00,1.00
3,Loss Prevention,800,1.00,0.53


In [130]:
comparison.sort_values(by=["Use Case", "Cutoff"])

,Use Case,Cutoff,Precision,Recall
0,Fraud Prevention,700,0.53,1.00
1,Fraud Prevention,800,1.00,1.00
2,Loss Prevention,700,1.00,1.00
3,Loss Prevention,800,1.00,0.53


In [132]:
comparison["Insight"] = [
    "More aggressive, catches all fraud but less precise",
    "Perfect fraud precision/recall",
    "Best overall for loss prevention",
    "Misses some losses"
]

comparison

,Use Case,Cutoff,Precision,Recall,Insight
0,Fraud Prevention,700,0.53,1.00,"More aggressive, catches all fraud but less pr..."
1,Fraud Prevention,800,1.00,1.00,Perfect fraud precision/recall
2,Loss Prevention,700,1.00,1.00,Best overall for loss prevention
3,Loss Prevention,800,1.00,0.53,Misses some losses


In [134]:
comparison = comparison[
    ["Use Case", "Cutoff", "Precision", "Recall", "Insight"]
]

In [139]:
pd.set_option("display.max_colwidth", None)

In [140]:
comparison

,Use Case,Cutoff,Precision,Recall,Insight
0,Fraud Prevention,700,0.53,1.00,"More aggressive, catches all fraud but less precise"
1,Fraud Prevention,800,1.00,1.00,Perfect fraud precision/recall
2,Loss Prevention,700,1.00,1.00,Best overall for loss prevention
3,Loss Prevention,800,1.00,0.53,Misses some losses


## Key Findings

**In this sample dataset, a cutoff of >= 700 perfectly separated bad outcomes from clear outcomes. A few important things to consider:** 
- At >= 800, fraud prevention looked perfect but loss prevention missed charged-off accounts.
- At >= 700, loss prevention achieved perfect precision and recall in this dataset.
- This suggests score bands may separate clear, charged-off, and fraud outcomes in the sample.
- Future work should test the cutoff on a larger and noisier dataset.